In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

words = open('/Users/wenduom/Desktop/ml/ml2026/makemore/names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)               
print(len(itos))

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [2]:
block_size = 6

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for c in w + '.':
            i = stoi[c]
            X.append(context)
            Y.append(i)        
            #print(''.join([itos[ii] for ii in context]) + ' --> ' + itos[i])
            context = context[1:] + [i]
                    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

    
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


torch.Size([182625, 6]) torch.Size([182625])
torch.Size([22655, 6]) torch.Size([22655])
torch.Size([22866, 6]) torch.Size([22866])


In [17]:
tt = torch.randn(4) * 0.01 # minimise initial loss by configuring weight to produce a smaller logit, try * 10, the loss will explode 
print(tt)
prob = torch.softmax(tt, dim=0)
print(prob)
loss = -prob[2].log()
print(loss)

tensor([-0.0200, -0.0192, -0.0004, -0.0053])
tensor([0.2478, 0.2480, 0.2527, 0.2515])
tensor(1.3755)


In [27]:
emb_dim = 12
n_hidden = 200
C = torch.rand((27, emb_dim))
W1 = torch.randn((emb_dim*block_size, n_hidden)) * (5/3)/((emb_dim*block_size)**0.5)
B1 = torch.randn(n_hidden)                       * 0.01
W2 = torch.randn((n_hidden, 27))                 * 0.01
B2 = torch.randn(27)                             * 0

bngain = torch.ones((1, n_hidden)) # batch normalisation gain
bnbias = torch.zeros((1, n_hidden))

bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, B1, W2, B2, bngain, bnbias]

In [28]:
tt = torch.randn(2,3)
print(tt)
print(tt.mean(0, keepdim=True)) # mean() test, it means across each dim/column

tensor([[0.5766, 0.4795, 0.2968],
        [0.2932, 0.0877, 0.4059]])
tensor([[0.4349, 0.2836, 0.3514]])


In [29]:
for p in parameters:
    p.requires_grad = True

batch_size = 32
lossi= []
for l in range(100000):
    # minibatch    
    ii = torch.randint(0, Xtr.shape[0], (batch_size,))
    xs = Xtr[ii]
    
    emb = C[xs].view(-1, emb_dim*block_size)
    hpreact = emb @ W1 + B1

    # batch normalization
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani)/bnstdi + bnbias

    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
    
    h = torch.tanh(hpreact) # (228146, 100)
    logits = h @ W2 + B2 # (228146, 27)
    loss = F.cross_entropy(logits, Ytr[ii])

    # reset grad
    for p in parameters:
        p.grad = None

    # back prop to compute the grad
    loss.backward()

    # update the weight
    for p in parameters:
        p.data += -0.01 * p.grad

    lossi.append(loss.log10().item())
    
    

In [30]:
with torch.no_grad():
    emb = C[Xtr]
    hpreact = emb.view(-1, emb_dim*block_size) @ W1 + B1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)

In [31]:
bnmean

tensor([[-0.6664,  0.4326,  0.9016,  0.2438, -0.4946, -0.3156, -0.5784,  0.3746,
          0.3160, -0.3976,  0.1974,  1.4751,  0.9111,  0.3555, -0.5605,  0.6845,
          0.0921,  0.4300, -0.9674, -0.7055, -0.1772, -0.2548,  0.4401,  0.5184,
          0.1088,  0.6887, -0.4204,  0.3799,  0.5198, -0.5565,  0.3272, -0.3352,
         -0.5008,  0.2960,  1.1804,  0.4011,  0.6948,  0.1816,  0.6218, -0.3933,
         -0.2921, -0.3559,  0.7047,  0.5794, -0.4873, -0.2431,  0.1678, -0.8537,
          0.9867,  0.1512, -0.6124, -0.5700,  0.2275,  0.3971,  0.3390,  0.7729,
          0.2230, -1.9683,  1.0944, -0.6052,  0.0888,  0.1216, -0.3255, -1.1007,
          0.7366,  0.2241,  1.2329,  0.4989,  0.8271,  0.6633,  0.5239,  1.4351,
         -0.4633, -0.4360, -0.2895, -0.4493,  0.9434,  0.4578, -0.7033,  0.0201,
          0.4827, -0.4963, -0.3788,  0.3377, -0.4281,  0.4326, -0.1218,  0.7645,
          1.2694,  0.1187,  0.0615, -0.2219,  0.1568,  0.7702, -0.2297,  0.4393,
          0.1190,  0.9288,  

In [32]:
bnmean_running

tensor([[-0.6613,  0.4197,  0.9168,  0.2415, -0.4784, -0.3241, -0.5951,  0.3804,
          0.3186, -0.4014,  0.1967,  1.4981,  0.9131,  0.3578, -0.5556,  0.6955,
          0.0926,  0.4293, -0.9634, -0.7120, -0.1789, -0.2664,  0.4561,  0.5205,
          0.0957,  0.7024, -0.4027,  0.3886,  0.5217, -0.5589,  0.3214, -0.3482,
         -0.4968,  0.3001,  1.1689,  0.4071,  0.6921,  0.1831,  0.6249, -0.3954,
         -0.2929, -0.3422,  0.7022,  0.5701, -0.4904, -0.2299,  0.1535, -0.8540,
          0.9848,  0.1465, -0.6051, -0.5757,  0.2134,  0.3895,  0.3456,  0.7799,
          0.2302, -1.9905,  1.0996, -0.6160,  0.0986,  0.1292, -0.3347, -1.1040,
          0.7203,  0.2265,  1.2450,  0.5070,  0.8386,  0.6485,  0.5158,  1.4396,
         -0.4565, -0.4300, -0.2940, -0.4726,  0.9422,  0.4451, -0.7018,  0.0216,
          0.4934, -0.5009, -0.3592,  0.3562, -0.4373,  0.4355, -0.1215,  0.7727,
          1.2775,  0.1147,  0.0717, -0.2166,  0.1601,  0.7736, -0.2215,  0.4422,
          0.1272,  0.9280,  

In [36]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte)
    }[split]
    emb = C[x].view(-1, emb_dim*block_size)
    hpreact = emb @ W1 + B1
    #hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True))/hpreact.std(0, keepdim=True) + bnbias
    #hpreact = bngain * (hpreact - bnmean)/bnstd + bnbias
    hpreact = bngain * (hpreact - bnmean_running)/bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + B2
    loss = F.cross_entropy(logits, y)
    print(split, loss)

split_loss('train')    
split_loss('val')    
    

train tensor(2.0925)
val tensor(2.1192)
